In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
project_path = '/content/drive/MyDrive/Fall 2024/EE 541'
import os
os.chdir(project_path)
print(f"Current working directory: {os.getcwd()}")

# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")

# print("Path to dataset files:", path)
# !cp -r /root/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/* /content/drive/MyDrive/EE541-project

Current working directory: /content/drive/MyDrive/Fall 2024/EE 541


In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from ast import literal_eval
from tqdm import tqdm
import plotly.express as px


In [9]:
#loading the dataset
keywords_df=pd.read_csv('keywords.csv')
movies_metadata_df=pd.read_csv('movies_metadata.csv',low_memory=False)
credit_df=pd.read_csv('credits.csv')

In [10]:
keywords_df[:1]

,id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."


In [11]:
credit_df[:1]

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862


In [12]:
movies_metadata_df[:1]

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0


In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Read the data
keywords_df = pd.read_csv('keywords.csv')
movies_metadata_df = pd.read_csv('movies_metadata.csv', low_memory=False)
credit_df = pd.read_csv('credits.csv')

# Select the required fields
movies_metadata_selected = movies_metadata_df[[
    'id', 'budget', 'revenue', 'popularity', 'genres', 'original_title',
    'overview', 'production_companies', 'production_countries',
    'spoken_languages', 'tagline', 'vote_average', 'vote_count'
]]

# Ensure the 'id' column has consistent data type (convert to string)
movies_metadata_selected['id'] = movies_metadata_selected['id'].astype(str)
keywords_df['id'] = keywords_df['id'].astype(str)

# Merge the data
merged_df = pd.merge(movies_metadata_selected, keywords_df[['id', 'keywords']], on='id', how='inner')

# Do not parse JSON formatted strings, keep the original format
merged_df['genres'] = merged_df['genres'].fillna('[]')
merged_df['keywords'] = merged_df['keywords'].fillna('[]')
merged_df['production_companies'] = merged_df['production_companies'].fillna('[]')
merged_df['production_countries'] = merged_df['production_countries'].fillna('[]')
merged_df['spoken_languages'] = merged_df['spoken_languages'].fillna('[]')

# Rearrange columns to move 'vote_average' and 'vote_count' to the end
columns_order = [
    'id', 'budget', 'revenue', 'popularity', 'genres', 'keywords',
    'original_title', 'overview', 'production_companies',
    'production_countries', 'spoken_languages', 'tagline',
    'vote_average', 'vote_count'
]
merged_df = merged_df[columns_order]

# Save the complete merged data as a CSV file
merged_df.to_csv('merged_movies_data.csv', index=False)
print("Merging completed, the full dataset has been saved as 'merged_movies_data.csv'")

# Split the data into training and testing sets with a 7:3 ratio
train_df, test_df = train_test_split(merged_df, test_size=0.3, random_state=42)

# Save the training and testing datasets
train_df.to_csv('movies_train.csv', index=False)
test_df.to_csv('movies_test.csv', index=False)

print("Data has been split into training and testing sets and saved as 'movies_train.csv' and 'movies_test.csv'")


/tmp/ipykernel_215/3814827954.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies_metadata_selected['id'] = movies_metadata_selected['id'].astype(str)


Merging completed, the full dataset has been saved as 'merged_movies_data.csv'
Data has been split into training and testing sets and saved as 'movies_train.csv' and 'movies_test.csv'


In [14]:
merged_df[:10]

,id,budget,revenue,popularity,genres,keywords,original_title,overview,production_companies,production_countries,spoken_languages,tagline,vote_average,vote_count
0,862,30000000,373554033.0,21.946943,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...",Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}]",NaN,7.7,5415.0
1,8844,65000000,262797249.0,17.015539,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'id': 10090, 'name': 'board game'}, {'id': 1...",Jumanji,When siblings Judy and Peter discover an encha...,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Roll the dice and unleash the excitement!,6.9,2413.0
2,15602,0,0.0,11.7129,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'id': 1495, 'name': 'fishing'}, {'id': 12392...",Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}]",Still Yelling. Still Fighting. Still Ready for...,6.5,92.0
3,31357,16000000,81452156.0,3.859495,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...","[{'id': 818, 'name': 'based on novel'}, {'id':...",Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}]",Friends are the people who let you be yourself...,6.1,34.0
4,11862,0,76578911.0,8.387519,"[{'id': 35, 'name': 'Comedy'}]","[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...",Father of the Bride Part II,Just when George Banks has recovered from his ...,"[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}]",Just When His World Is Back To Normal... He's ...,5.7,173.0
5,949,60000000,187436818.0,17.924927,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...","[{'id': 642, 'name': 'robbery'}, {'id': 703, '...",Heat,"Obsessive master thief, Neil McCauley leads a ...","[{'name': 'Regency Enterprises', 'id': 508}, {...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",A Los Angeles Crime Saga,7.7,1886.0
6,11860,58000000,0.0,6.677277,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '...","[{'id': 90, 'name': 'paris'}, {'id': 380, 'nam...",Sabrina,An ugly duckling having undergone a remarkable...,"[{'name': 'Paramount Pictures', 'id': 4}, {'na...","[{'iso_3166_1': 'DE', 'name': 'Germany'}, {'is...","[{'iso_639_1': 'fr', 'name': 'Français'}, {'is...",You are cordially invited to the most surprisi...,6.2,141.0
7,45325,0,0.0,2.561161,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",[],Tom and Huck,"A mischievous young boy, Tom Sawyer, witnesses...","[{'name': 'Walt Disney Pictures', 'id': 2}]","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",The Original Bad Boys.,5.4,45.0
8,9091,35000000,64350171.0,5.23158,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'id': 949, 'name': 'terrorist'}, {'id': 1562...",Sudden Death,International action superstar Jean Claude Van...,"[{'name': 'Universal Pictures', 'id': 33}, {'n...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}]",Terror goes into overtime.,5.5,174.0
9,710,58000000,352194034.0,14.686036,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...","[{'id': 701, 'name': 'cuba'}, {'id': 769, 'nam...",GoldenEye,James Bond must unmask the mysterious head of ...,"[{'name': 'United Artists', 'id': 60}, {'name'...","[{'iso_3166_1': 'GB', 'name': 'United Kingdom'...","[{'iso_639_1': 'en', 'name': 'English'}